# Logic-driven consistency: fuzzy-logic losses for NLI

This notebook implements the core ideas of [*A Logic-Driven Framework for Consistency of Neural Models*](https://arxiv.org/abs/1909.00126) (Li, Gupta, Mehta and Srikumar, EMNLP 2019) with `symprog`.

The paper starts from an observation: accurate models can still be **inconsistent**. A BERT model with about 90% accuracy on SNLI predicts that a premise $P$ entails $H$, and that $H$ contradicts $Z$, but then fails to predict that $P$ contradicts $Z$ in about 46% of such cases. The paper's answer has two parts:

- **Measure** inconsistency by writing the invariants of the task as logical rules over model predictions, and counting how often they are violated. Classification error is the special case where the only rules are the gold labels.
- **Reduce** it by relaxing the rules into differentiable losses with a **t-norm** (fuzzy logic), and adding them to the training loss. The rules apply to unlabeled data, so consistency can be learned from examples nobody annotated.

`symprog` represents constraints as Boolean formulas and reasons about them *semantically*, through their models (weighted model counting, Viterbi, Gödel-style fuzzy inference over models). The paper's fuzzy logic is different: it relaxes each connective of the formula *as written*. So the main piece of this notebook is a small translator from a native `symprog` formula into the paper's fuzzy logic.

This notebook covers:

1. **The task**: natural language inference, with examples from SNLI on the Hugging Face hub.
2. **The model**: an off-the-shelf encoder (cross-encoder) for NLI, and its predictions.
3. **Consistency in logic**: the paper's symmetry and transitivity rules as `symprog` formulas, and the paper's violation metrics, measured on real model predictions.
4. **From formulas to fuzzy losses**: translating a `symprog` formula into product, Gödel or Łukasiewicz fuzzy logic, and recovering the paper's losses.
5. **Fuzzy versus probabilistic semantics**: how the t-norm relaxation compares with `symprog`'s weighted model counting.
6. **Training for consistency**: the combined loss, a small repair example, and what the paper found.

## Setup

Uncomment the install line if you are running on Colab. If `symprog` is not installed, the notebook uses the version in this repository. The NLI sections also need `transformers` and `datasets`, and download a small model (about 330 MB) and the SNLI validation set the first time they run.

In [1]:
# %pip install -q "symprog[notebooks]" transformers datasets

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import ast
import itertools
import math
import warnings
from collections import defaultdict

import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from symprog import Solver

warnings.filterwarnings("ignore", category=symprog.ExperimentalWarning)
_ = torch.manual_seed(0)

# Natural language inference

In **natural language inference** (NLI), a model reads a *premise* $P$ and a *hypothesis* $H$, and decides whether $P$ **entails** $H$ (E), **contradicts** it (C), or neither (**neutral**, N). The paper trains on two standard datasets:

- [SNLI](https://huggingface.co/datasets/stanfordnlp/snli) (Bowman et al., 2015): 570k pairs, with premises taken from image captions.
- [MultiNLI](https://huggingface.co/datasets/nyu-mll/multi_nli) (Williams et al., 2018): 433k pairs from ten genres of text.

Both are on the Hugging Face hub. SNLI's validation split is small enough to load here:

In [2]:
snli = load_dataset("stanfordnlp/snli", split="validation")
LABEL_NAMES = snli.features["label"].names   ## ['entailment', 'neutral', 'contradiction']
print(snli)
print("labels:", LABEL_NAMES)

for example in snli.select(range(6)):
    print(f"\nP: {example['premise']}\nH: {example['hypothesis']}\n   -> {LABEL_NAMES[example['label']]}")

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10000
})
labels: ['entailment', 'neutral', 'contradiction']

P: Two women are embracing while holding to go packages.
H: The sisters are hugging goodbye while holding to go packages after just eating lunch.
   -> neutral

P: Two women are embracing while holding to go packages.
H: Two woman are holding packages.
   -> entailment

P: Two women are embracing while holding to go packages.
H: The men are fighting outside a deli.
   -> contradiction

P: Two young children in blue jerseys, one with the number 9 and one with the number 2 are standing on wooden steps in a bathroom and washing their hands in a sink.
H: Two kids in numbered jerseys wash their hands.
   -> entailment

P: Two young children in blue jerseys, one with the number 9 and one with the number 2 are standing on wooden steps in a bathroom and washing their hands in a sink.
H: Two kids at a ballgame wash their hands.
   -> neutral

P: Two young childr

SNLI was built by showing annotators a premise (a caption) and asking for one hypothesis per label. So each premise comes with a group of hypotheses, one entailed, one neutral and one contradicted. Some pairs have no gold label (`-1`), because the annotators disagreed.

In [3]:
groups = defaultdict(list)
for example in snli:
    if example["label"] != -1:
        groups[example["premise"]].append((example["hypothesis"], LABEL_NAMES[example["label"]]))

premise, hypotheses = next(iter(groups.items()))
print("P:", premise)
for hypothesis, label in hypotheses:
    print(f"   {label:13} {hypothesis}")
print(f"\n{len(groups)} premises, {sum(len(h) for h in groups.values())} labeled pairs")

P: Two women are embracing while holding to go packages.
   neutral       The sisters are hugging goodbye while holding to go packages after just eating lunch.
   entailment    Two woman are holding packages.
   contradiction The men are fighting outside a deli.

3319 premises, 9842 labeled pairs


# An encoder model for NLI

The paper's models are BERT$_\text{base}$ fine-tuned on SNLI and MultiNLI (and an LSTM baseline). BERT is used as a **cross-encoder**: the premise and hypothesis are joined into one input, `[CLS] P [SEP] H [SEP]`, and a classification layer on the `[CLS]` representation outputs the three label scores.

We use a small, publicly available model of the same kind, [`cross-encoder/nli-distilroberta-base`](https://huggingface.co/cross-encoder/nli-distilroberta-base): a DistilRoBERTa encoder fine-tuned on SNLI and MultiNLI. Its labels are in its own order, so we read them from its config and return probabilities in the order (E, C, N).

In [4]:
MODEL_NAME = "cross-encoder/nli-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).eval()

LABELS = ["E", "C", "N"]
by_name = {name.lower(): i for i, name in model.config.id2label.items()}
ECN = [by_name["entailment"], by_name["contradiction"], by_name["neutral"]]
print("model labels:", model.config.id2label, "-> reordered as (E, C, N):", ECN)

def nli_logits(premises, hypotheses, model=model):
    "The model's label scores for a batch of pairs, as a tensor [batch, 3] in the order (E, C, N)."
    inputs = tokenizer(premises, hypotheses, padding=True, truncation=True, return_tensors="pt")
    return model(**inputs).logits[:, ECN]

@torch.no_grad()
def nli_probs(premises, hypotheses, batch_size=64):
    "Label probabilities for many pairs, without gradients."
    out = [torch.softmax(nli_logits(premises[i:i + batch_size], hypotheses[i:i + batch_size]), dim=-1)
           for i in range(0, len(premises), batch_size)]
    return torch.cat(out)

def show(pairs):
    probs = nli_probs([p for p, _ in pairs], [h for _, h in pairs])
    for (p, h), pr in zip(pairs, probs):
        dist = "  ".join(f"{l}={x:.2f}" for l, x in zip(LABELS, pr.tolist()))
        print(f"{LABELS[pr.argmax()]}  ({dist})   P: {p}  |  H: {h}")

model labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'} -> reordered as (E, C, N): [1, 0, 2]


The paper's opening example is a triple of sentences: $P$ entails $H$, and $H$ contradicts $Z$, so $P$ must contradict $Z$. Here is what our model predicts for the three pairs, and for the pairs with premise and hypothesis swapped:

In [5]:
P = "John is on a train to Berlin."
H = "John is traveling to Berlin."
Z = "John is having lunch in Berlin."

show([(P, H), (H, Z), (P, Z)])
print()
show([(H, P), (Z, H), (Z, P)])

E  (E=0.97  C=0.00  N=0.03)   P: John is on a train to Berlin.  |  H: John is traveling to Berlin.
C  (E=0.00  C=0.59  N=0.41)   P: John is traveling to Berlin.  |  H: John is having lunch in Berlin.
C  (E=0.00  C=0.79  N=0.21)   P: John is on a train to Berlin.  |  H: John is having lunch in Berlin.

N  (E=0.01  C=0.00  N=0.99)   P: John is traveling to Berlin.  |  H: John is on a train to Berlin.
N  (E=0.22  C=0.15  N=0.62)   P: John is having lunch in Berlin.  |  H: John is traveling to Berlin.
C  (E=0.00  C=0.99  N=0.01)   P: John is having lunch in Berlin.  |  H: John is on a train to Berlin.


The model gets the triple right, but it is not consistent across directions: it predicts contradiction for $(H, Z)$ and neutral for $(Z, H)$, although contradiction should be symmetric. Predictions on related pairs are not independent: some combinations of labels are impossible. The next section writes these invariants down in logic.

# Consistency in logic

The paper writes everything a model should satisfy as a conjunction of rules $\bigwedge_{(L, R)} L(x) \rightarrow R(x)$, where $L$ and $R$ are Boolean formulas over the model's predictions on a group of examples $x$. The predicate $E(P, H)$ means "the model predicts entailment for $(P, H)$", and similarly for $C$ and $N$.

- **Annotation consistency.** A labeled example $(P, H)$ with gold label $Y^\star$ is the rule $\top \rightarrow Y^\star(P, H)$.
- **Symmetry consistency.** Contradiction is symmetric: $C(P, H) \leftrightarrow C(H, P)$.
- **Transitivity consistency.** For three sentences, not all $3^3 = 27$ label assignments to $(P,H)$, $(H,Z)$ and $(P,Z)$ are allowed:

$$\begin{aligned}
&(E(P,H) \wedge E(H,Z) \rightarrow E(P,Z)) \;\wedge\; (E(P,H) \wedge C(H,Z) \rightarrow C(P,Z)) \\
\wedge\; &(N(P,H) \wedge E(H,Z) \rightarrow \neg C(P,Z)) \;\wedge\; (N(P,H) \wedge C(H,Z) \rightarrow \neg E(P,Z))
\end{aligned}$$

In `symprog`, these are ordinary formula strings. We name the variable for "label $l$ for pair $(a, b)$" as `l_ab`, e.g., `C_PH`:

In [6]:
def var(label, a, b):
    return f"{label}_{a}{b}"

def exactly_one_label(a, b):
    "Each pair gets exactly one label (`Xor` with several arguments means exactly one)."
    return f"Xor({', '.join(var(l, a, b) for l in LABELS)})"

SYMMETRY = f"Biconditional({var('C', 'P', 'H')}, {var('C', 'H', 'P')})"

TRANSITIVITY = [
    f"Implies(And({var('E','P','H')}, {var('E','H','Z')}), {var('E','P','Z')})",
    f"Implies(And({var('E','P','H')}, {var('C','H','Z')}), {var('C','P','Z')})",
    f"Implies(And({var('N','P','H')}, {var('E','H','Z')}), Not({var('C','P','Z')}))",
    f"Implies(And({var('N','P','H')}, {var('C','H','Z')}), Not({var('E','P','Z')}))",
]
for rule in [SYMMETRY, *TRANSITIVITY]:
    print(rule)

Biconditional(C_PH, C_HP)
Implies(And(E_PH, E_HZ), E_PZ)
Implies(And(E_PH, C_HZ), C_PZ)
Implies(And(N_PH, E_HZ), Not(C_PZ))
Implies(And(N_PH, C_HZ), Not(E_PZ))


Because the rules are formulas, `symprog` can reason about them directly. For example, model counting tells us how many of the 27 label assignments to a triple the transitivity rules allow, and we can list the ones they forbid:

In [7]:
PAIRS = [("P", "H"), ("H", "Z"), ("P", "Z")]
labelings = [exactly_one_label(a, b) for a, b in PAIRS]
triple_vars = [var(l, a, b) for a, b in PAIRS for l in LABELS]

def count(formulas):
    solver = Solver()
    solver.add(formulas)
    solver.add_weights({v: 0.5 for v in triple_vars})   ## Count ignores the weights
    return int(solver.query(mode="Count").inference)

print("label assignments:              ", count(labelings))
print("allowed by the transitivity rules:", count(labelings + TRANSITIVITY))

def holds(formula, assignment):
    "Evaluates a formula string under a 0/1 assignment (a dict from variables to bools)."
    return eval(compile(ast.parse(formula, mode="eval"), "<formula>", "eval"),
                {"And": lambda *x: all(x), "Or": lambda *x: any(x), "Not": lambda x: not x,
                 "Implies": lambda x, y: (not x) or y, "Biconditional": lambda x, y: x == y,
                 "Xor": lambda *x: sum(x) == 1}, dict(assignment))

forbidden = []
for labels in itertools.product(LABELS, repeat=3):
    assignment = {var(l, a, b): l == chosen for (a, b), chosen in zip(PAIRS, labels) for l in LABELS}
    if not all(holds(rule, assignment) for rule in TRANSITIVITY):
        forbidden.append("".join(labels))
print("forbidden (labels of PH, HZ, PZ):", forbidden)

label assignments:               27
allowed by the transitivity rules: 21
forbidden (labels of PH, HZ, PZ): ['EEC', 'EEN', 'ECE', 'ECN', 'NEC', 'NCE']


#### Measuring inconsistency

The paper defines two error rates over a dataset $D$ of example groups $x$, using the model's (argmax) predictions:

$$\rho = \frac{\sum_{x \in D} \big[\bigvee_{(L,R)} \neg (L(x) \rightarrow R(x))\big]}{|D|}
\qquad
\tau = \frac{\sum_{x \in D} \big[\bigvee_{(L,R)} \neg (L(x) \rightarrow R(x))\big]}{\sum_{x \in D} \big[\bigvee_{(L,R)} L(x)\big]}$$

The **global** violation $\rho$ is the fraction of groups that violate some rule. The **conditional** violation $\tau$ only counts groups where some rule's antecedent holds, since a rule whose antecedent is false is trivially satisfied. For annotation rules ($\top \rightarrow Y^\star$) both equal the classification error.

To measure symmetry, the paper swaps premise and hypothesis (**mirrored** instances). Here we do this for 500 SNLI validation pairs:

In [8]:
def predicted(probs, prefix_pairs):
    "0/1 assignments from argmax predictions: probs[i] holds the (E, C, N) distributions of the pairs in `prefix_pairs`."
    out = []
    for row in probs:
        assignment = {}
        for (a, b), p in zip(prefix_pairs, row):
            for l, i in zip(LABELS, range(3)):
                assignment[var(l, a, b)] = bool(p.argmax() == i)
        out.append(assignment)
    return out

def violations(rules, assignments):
    "The paper's global (rho) and conditional (tau) violation rates of rules written as Implies(L, R) or Biconditional."
    def antecedents(rule):
        tree = ast.parse(rule, mode="eval").body
        if tree.func.id == "Implies":
            return [ast.unparse(tree.args[0])]
        return [ast.unparse(a) for a in tree.args]      ## Biconditional(A, B) = (A -> B) and (B -> A)
    violated = [any(not holds(r, x) for r in rules) for x in assignments]
    triggered = [any(holds(a, x) for r in rules for a in antecedents(r)) for x in assignments]
    return sum(violated) / len(assignments), sum(violated) / max(1, sum(triggered))

examples = [ex for ex in snli.select(range(500)) if ex["label"] != -1]
pairs = [(ex["premise"], ex["hypothesis"]) for ex in examples]
forward = nli_probs([p for p, _ in pairs], [h for _, h in pairs])
mirrored = nli_probs([h for _, h in pairs], [p for p, _ in pairs])

sym = predicted(torch.stack([forward, mirrored], dim=1), [("P", "H"), ("H", "P")])
rho_s, tau_s = violations([SYMMETRY], sym)
gold = [LABEL_NAMES[ex["label"]][0].upper() for ex in examples]      ## 'entailment' -> 'E', ...
accuracy = sum(LABELS[p.argmax()] == g for p, g in zip(forward, gold)) / len(pairs)
print(f"{len(pairs)} pairs: accuracy {accuracy:.3f}   symmetry violations: rho_S = {rho_s:.3f}, tau_S = {tau_s:.3f}")

490 pairs: accuracy 0.910   symmetry violations: rho_S = 0.131, tau_S = 0.317


For transitivity we need triples of related sentences. The paper samples 100k triples of MS COCO captions about the same topic. A cheap alternative is SNLI itself: the hypotheses written for the same premise are about the same scene, so $(P, H_i, H_j)$ for two hypotheses of one premise gives three related pairs, two of them with gold labels.

In [9]:
triples = [(p, hs[i][0], hs[j][0]) for p, hs in itertools.islice(groups.items(), 200)
           for i, j in itertools.permutations(range(len(hs)), 2)]
unique = sorted({pair for (p, h, z) in triples for pair in [(p, h), (h, z), (p, z)]})
probs = dict(zip(unique, nli_probs([a for a, _ in unique], [b for _, b in unique])))

triple_probs = torch.stack([torch.stack([probs[(p, h)], probs[(h, z)], probs[(p, z)]]) for p, h, z in triples])
tra = predicted(triple_probs, PAIRS)
rho_t, tau_t = violations(TRANSITIVITY, tra)
print(f"{len(triples)} triples: transitivity violations: rho_T = {rho_t:.3f}, tau_T = {tau_t:.3f}")

for rule in TRANSITIVITY:
    r, t = violations([rule], tra)
    print(f"   rho = {r:.3f}  tau = {t:.3f}   {rule}")

1160 triples: transitivity violations: rho_T = 0.029, tau_T = 0.074
   rho = 0.002  tau = 0.133   Implies(And(E_PH, E_HZ), E_PZ)
   rho = 0.016  tau = 0.095   Implies(And(E_PH, C_HZ), C_PZ)
   rho = 0.000  tau = 0.000   Implies(And(N_PH, E_HZ), Not(C_PZ))
   rho = 0.012  tau = 0.082   Implies(And(N_PH, C_HZ), Not(E_PZ))


For comparison, the paper's BERT$_\text{base}$ trained on all of SNLI and MultiNLI has $\rho_S = 18.1\%$, $\tau_S = 59.6\%$, $\rho_T = 4.5\%$ and $\tau_T = 14.8\%$ on its own evaluation sets (Table 2). Our numbers come from a different model and different (easier, partly labeled) triples, so they are not directly comparable, but the pattern of how often rules are violated is the point.

# From formulas to fuzzy losses

To train with these rules, the paper relaxes them into differentiable functions of the model's **probabilities**: $e(P,H)$, $c(P,H)$ and $n(P,H)$ replace the Boolean predicates. Each connective becomes a function on $[0, 1]$, given by a **t-norm** (Table 1 of the paper):

| | Boolean | Product | Gödel | Łukasiewicz |
|---|---|---|---|---|
| Negation | $\neg A$ | $1 - a$ | $1 - a$ | $1 - a$ |
| T-norm | $A \wedge B$ | $ab$ | $\min(a, b)$ | $\max(0, a + b - 1)$ |
| T-conorm | $A \vee B$ | $a + b - ab$ | $\max(a, b)$ | $\min(1, a + b)$ |
| Residuum | $A \rightarrow B$ | $\min(1, b / a)$ | $1$ if $b \geq a$, else $b$ | $\min(1, 1 - a + b)$ |

The paper uses the **product** t-norm, and takes the negative logarithm of the relaxed formula as the loss.

#### A translator for symprog formulas

`symprog` does not implement this kind of fuzzy logic. Its `Fuzzy` mode is a *semiring*: it computes $\max$ over the formula's models of the $\min$ of their literal weights, on the compiled circuit. That depends only on the formula's meaning, not on how it is written. The paper's relaxation instead follows the **syntax** of the formula, connective by connective.

So we write a translator. It parses a formula string with the same grammar `symprog` uses (`And`, `Or`, `Not`, `Xor`, `Implies`, `Biconditional`, `True`, `False`, and the operators `~`, `&`, `|`), and returns a function from truth values (tensors in $[0, 1]$) to the relaxed truth value of the formula. `Biconditional(A, B)` is read as $(A \rightarrow B) \wedge (B \rightarrow A)$, and `Xor` with several arguments as "exactly one".

In [10]:
class TNorm:
    "The four operations of a t-norm fuzzy logic, on tensors in [0, 1]."
    def __init__(self, name, t_norm, t_conorm, residuum):
        self.name, self.t_norm, self.t_conorm, self.residuum = name, t_norm, t_conorm, residuum
    def negation(self, a):
        return 1 - a

EPS = 1e-12
PRODUCT = TNorm("product",
    t_norm=lambda a, b: a * b,
    t_conorm=lambda a, b: a + b - a * b,
    residuum=lambda a, b: torch.where(a <= b, torch.ones_like(b), b / torch.clamp(a, min=EPS)))
GODEL = TNorm("Gödel",
    t_norm=torch.minimum,
    t_conorm=torch.maximum,
    residuum=lambda a, b: torch.where(b >= a, torch.ones_like(b), b))
LUKASIEWICZ = TNorm("Łukasiewicz",
    t_norm=lambda a, b: torch.clamp(a + b - 1, min=0.0),
    t_conorm=lambda a, b: torch.clamp(a + b, max=1.0),
    residuum=lambda a, b: torch.clamp(1 - a + b, max=1.0))

def to_fuzzy(formula, logic=PRODUCT):
    "Translates a symprog formula string into a function `values -> truth value` in the fuzzy `logic`."
    tree = ast.parse(formula, mode="eval").body

    def build(node):
        if isinstance(node, ast.Name):
            if node.id in ("True", "False"):
                return lambda v: torch.tensor(1.0 if node.id == "True" else 0.0)
            return lambda v: v[node.id]
        if isinstance(node, ast.Constant) and isinstance(node.value, bool):
            return lambda v: torch.tensor(float(node.value))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.Invert, ast.Not)):
            arg = build(node.operand)
            return lambda v: logic.negation(arg(v))
        if isinstance(node, ast.BinOp) and isinstance(node.op, (ast.BitAnd, ast.BitOr)):
            name = "And" if isinstance(node.op, ast.BitAnd) else "Or"
            return build(ast.Call(func=ast.Name(id=name), args=[node.left, node.right], keywords=[]))
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            args = [build(a) for a in node.args]
            op = node.func.id
            if op == "Not":
                return lambda v: logic.negation(args[0](v))
            if op in ("And", "Or"):
                combine = logic.t_norm if op == "And" else logic.t_conorm
                def nary(v):
                    out = args[0](v)
                    for a in args[1:]:
                        out = combine(out, a(v))
                    return out
                return nary
            if op == "Implies":
                return lambda v: logic.residuum(args[0](v), args[1](v))
            if op == "Biconditional":
                return lambda v: logic.t_norm(logic.residuum(args[0](v), args[1](v)),
                                              logic.residuum(args[1](v), args[0](v)))
            if op == "Xor":   ## exactly one: Or over i of (x_i and not x_j for all j != i)
                def exactly_one(v):
                    xs = [a(v) for a in args]
                    terms = []
                    for i, x in enumerate(xs):
                        term = x
                        for j, y in enumerate(xs):
                            if j != i:
                                term = logic.t_norm(term, logic.negation(y))
                        terms.append(term)
                    out = terms[0]
                    for t in terms[1:]:
                        out = logic.t_conorm(out, t)
                    return out
                return exactly_one
        raise ValueError(f"unsupported syntax in formula: {ast.unparse(node)}")

    return build(tree)

def fuzzy_loss(formulas, values, logic=PRODUCT):
    "The paper's loss: -log of the relaxed truth value of the conjunction of `formulas` (summed over the batch)."
    formulas = [formulas] if isinstance(formulas, str) else formulas
    truth = to_fuzzy(f"And({', '.join(formulas)})" if len(formulas) > 1 else formulas[0], logic)(values)
    return 0.0 - torch.log(torch.clamp(truth, min=EPS)).sum()

On 0/1 values, every t-norm agrees with Boolean logic, so the translator can be checked against the Boolean evaluation of each rule, on all 27 label assignments of a triple:

In [11]:
for logic in [PRODUCT, GODEL, LUKASIEWICZ]:
    for labels in itertools.product(LABELS, repeat=3):
        boolean = {var(l, a, b): l == chosen for (a, b), chosen in zip(PAIRS, labels) for l in LABELS}
        values = {k: torch.tensor(float(x)) for k, x in boolean.items()}
        for rule in [*TRANSITIVITY, *labelings]:
            assert to_fuzzy(rule, logic)(values).item() == float(holds(rule, boolean))
print("all t-norms agree with Boolean logic on 0/1 values")

all t-norms agree with Boolean logic on 0/1 values


#### Recovering the paper's losses

With the product t-norm, the translator gives back the losses derived in the paper.

**Annotation consistency is cross-entropy.** The rule $\top \rightarrow Y^\star(P, H)$ relaxes to $\min(1, y^\star / 1) = y^\star$, so the loss is $-\log y^\star(P, H)$, the usual cross-entropy (eq. 9).

In [12]:
p = torch.softmax(torch.randn(4, 3), dim=-1)          ## a batch of 4 predicted distributions over (E, C, N)
gold = torch.tensor([0, 1, 2, 1])
values = {"E_PH": p[:, 0], "C_PH": p[:, 1], "N_PH": p[:, 2]}

annotation_rules = [f"Implies(True, {LABELS[g]}_PH)" for g in gold]
annotation = sum(fuzzy_loss(rule, {k: v[i] for k, v in values.items()}) for i, rule in enumerate(annotation_rules))
cross_entropy = torch.nn.functional.nll_loss(torch.log(p), gold, reduction="sum")
print(f"fuzzy annotation loss {annotation.item():.4f}   cross-entropy {cross_entropy.item():.4f}")
assert torch.isclose(annotation, cross_entropy)

fuzzy annotation loss 5.7646   cross-entropy 5.7646


**Symmetry.** $C(P,H) \leftrightarrow C(H,P)$ relaxes to $\min\big(1, \frac{c(H,P)}{c(P,H)}\big)\min\big(1, \frac{c(P,H)}{c(H,P)}\big)$, whose negative logarithm is

$$L_\text{sym} = \big|\log c(P,H) - \log c(H,P)\big| \qquad \text{(eq. 12)}$$

**Transitivity.** A rule $A \wedge B \rightarrow C$ relaxes to $\min\big(1, \frac{c}{ab}\big)$, so its loss is $\max\big(0, \log a + \log b - \log c\big)$: it is zero when the conclusion is at least as likely as the premises together, and grows with the gap. A negated conclusion $\neg C$ uses $\log(1 - c)$ instead.

In [13]:
c_ph = torch.tensor([0.9, 0.2, 0.6, 0.05])
c_hp = torch.tensor([0.8, 0.7, 0.6, 0.90])
sym_loss = fuzzy_loss(SYMMETRY, {"C_PH": c_ph, "C_HP": c_hp})
paper = (torch.log(c_ph) - torch.log(c_hp)).abs().sum()
print(f"translated symmetry loss {sym_loss.item():.4f}   paper's eq. 12 {paper.item():.4f}")
assert torch.isclose(sym_loss, paper)

a = torch.tensor([0.9, 0.9, 0.5, 0.1])                ## e(P,H)
b = torch.tensor([0.8, 0.8, 0.5, 0.9])                ## c(H,Z)
c = torch.tensor([0.9, 0.3, 0.1, 0.2])                ## c(P,Z)
rule = TRANSITIVITY[1]                                 ## E(P,H) and C(H,Z) -> C(P,Z)
per_triple = [fuzzy_loss(rule, {"E_PH": a[i], "C_HZ": b[i], "C_PZ": c[i]}).item() for i in range(4)]
paper = torch.relu(torch.log(a) + torch.log(b) - torch.log(c))
for i in range(4):
    print(f"a={a[i]:.1f} b={b[i]:.1f} c={c[i]:.1f}:  translated loss {per_triple[i]:.4f}   max(0, log a + log b - log c) {paper[i]:.4f}")
assert torch.allclose(torch.tensor(per_triple), paper, atol=1e-5)

translated symmetry loss 4.2609   paper's eq. 12 4.2609
a=0.9 b=0.8 c=0.9:  translated loss 0.0000   max(0, log a + log b - log c) 0.0000
a=0.9 b=0.8 c=0.3:  translated loss 0.8755   max(0, log a + log b - log c) 0.8755
a=0.5 b=0.5 c=0.1:  translated loss 0.9163   max(0, log a + log b - log c) 0.9163
a=0.1 b=0.9 c=0.2:  translated loss 0.0000   max(0, log a + log b - log c) 0.0000


On the model's predictions for the Berlin example, the losses are computed from the probabilities of the three pairs. The choice of t-norm changes the numbers, and (as the paper notes) the gradients: Gödel logic passes gradient to a single argument, and Łukasiewicz logic gives zero gradient whenever a relaxed rule is already fully satisfied.

In [14]:
berlin = nli_probs([P, H, P], [H, Z, Z])
berlin_values = {var(l, a, b): berlin[k, i] for k, (a, b) in enumerate(PAIRS) for i, l in enumerate(LABELS)}
for logic in [PRODUCT, GODEL, LUKASIEWICZ]:
    print(f"{logic.name:12} transitivity loss: {fuzzy_loss(TRANSITIVITY, berlin_values, logic).item():.4f}")

product      transitivity loss: 0.2633
Gödel        transitivity loss: 6.1271
Łukasiewicz  transitivity loss: 0.0000


# Fuzzy versus probabilistic semantics

The translator relaxes formulas **syntactically**, so logically equivalent formulas can get different losses. For example, $A \rightarrow B$ and $\neg A \vee B$ are the same Boolean function, but with the product t-norm the first becomes the residuum $\min(1, b/a)$ and the second the "S-implication" $1 - a + ab$.

`symprog`'s inference is **semantic**: it depends only on the models of the formula. With `LogProb`, the formula's probability when each variable is true independently with its weight is a weighted model count, and its negative is the *semantic loss* (see `semantic_loss.ipynb`). For a single implication this happens to be the S-implication: $\Pr(\neg A \vee B) = 1 - a + ab$. `symprog`'s `Fuzzy` mode is semantic too, the max over models of the min literal weight.

In [15]:
def symprog_inference(formula, weights, mode):
    solver = Solver()
    solver.add([formula])
    solver.add_weights(weights)
    return solver.query(mode=mode).inference.item()

weights = {"a": 0.9, "b": 0.3}
values = {k: torch.tensor(v) for k, v in weights.items()}
print(f"{'formula':22} {'product':>8} {'Gödel':>8} {'Łukas.':>8} {'WMC':>8} {'symprog Fuzzy':>14}")
for formula in ["Implies(a, b)", "Or(Not(a), b)", "Not(And(a, Not(b)))"]:
    row = [to_fuzzy(formula, logic)(values).item() for logic in [PRODUCT, GODEL, LUKASIEWICZ]]
    row += [symprog_inference(formula, weights, "Prob"), symprog_inference(formula, weights, "Fuzzy")]
    print(f"{formula:22} " + " ".join(f"{x:8.4f}" for x in row[:4]) + f" {row[4]:14.4f}")

formula                 product    Gödel   Łukas.      WMC  symprog Fuzzy
Implies(a, b)            0.3333   0.3000   0.4000   0.3700         0.3000
Or(Not(a), b)            0.3700   0.3000   0.4000   0.3700         0.3000
Not(And(a, Not(b)))      0.3700   0.3000   0.4000   0.3700         0.3000


Only the last two columns are guaranteed to be the same for all three ways of writing the formula. Here the product logic gives a different value for `Implies`. In Łukasiewicz logic the residuum $\min(1, 1 - a + b)$ happens to equal its relaxation of $\neg A \vee B$, and in Gödel logic the values coincide for these weights but not in general: for $a = 0.3, b = 0.2$ the residuum gives $0.2$ and $\max(1 - a, b)$ gives $0.7$. This is one of the trade-offs discussed in the course: t-norm relaxations are cheap (linear in the size of the formula, with no compilation) and keep each rule's gradient simple, while semantic approaches respect logical equivalence but require model counting, which is hard in general. For the paper's rules, which are small and grounded one triple at a time, both are feasible.

One more difference matters for NLI: the three label probabilities of a pair come from a softmax, so they are not independent. The t-norm losses simply plug in $e$, $c$ and $n$, while a probabilistic treatment would condition on each pair having exactly one label.

# Training for consistency

The paper's training objective adds the relaxed rules to cross-entropy:

$$L = L_\text{ann} + \lambda_\text{sym} L_\text{sym} + \lambda_\text{tran} L_\text{tran}$$

$L_\text{ann}$ uses labeled pairs. The other two losses need **no labels**: the paper uses mirrored training pairs (**M**), 100k mirrored pairs from MS COCO captions (**U**), and 100k caption triples (**T**). The constrained models start from the fine-tuned BERT and are fine-tuned again with the combined loss; the $\lambda$'s are chosen by development accuracy.

#### Repairing a prediction

To see what the transitivity loss does, take the model's predicted logits for a triple that violates a rule, treat them as free parameters, and minimize the transitivity loss plus a penalty for moving away from the original prediction (as in the path-repair example of `semantic_loss.ipynb`). Training does the same thing to the network's parameters, over many triples at once.

In [16]:
violating = [k for k, x in enumerate(tra) if not all(holds(rule, x) for rule in TRANSITIVITY)]
if violating:
    p_, h_, z_ = triples[violating[0]]
else:
    p_, h_, z_ = P, H, Z
with torch.no_grad():
    original = nli_logits([p_, h_, p_], [h_, z_, z_])

def triple_values(probs):
    return {var(l, a, b): probs[k, i] for k, (a, b) in enumerate(PAIRS) for i, l in enumerate(LABELS)}

def describe(probs):
    return "  ".join(f"{a}{b}: {LABELS[probs[k].argmax()]} ({probs[k].max():.2f})" for k, (a, b) in enumerate(PAIRS))

logits = original.clone().requires_grad_(True)
optimizer = torch.optim.Adam([logits], lr=0.05)
for step in range(300):
    probs = torch.softmax(logits, dim=-1)
    loss = fuzzy_loss(TRANSITIVITY, triple_values(probs)) + 0.05 * ((logits - original) ** 2).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

before, after = torch.softmax(original, dim=-1), torch.softmax(logits.detach(), dim=-1)
print(f"P: {p_}\nH: {h_}\nZ: {z_}\n")
print("before:", describe(before), "  consistent:", all(holds(r, predicted(before[None], PAIRS)[0]) for r in TRANSITIVITY))
print("after: ", describe(after), "  consistent:", all(holds(r, predicted(after[None], PAIRS)[0]) for r in TRANSITIVITY))

P: Two women having drinks and smoking cigarettes at the bar.
H: Two women are at a bar.
Z: Three women are at a bar.

before: PH: E (0.99)  HZ: C (0.97)  PZ: E (0.55)   consistent: False


after:  PH: E (0.98)  HZ: C (0.86)  PZ: C (0.84)   consistent: True


#### The combined loss for fine-tuning

The paper's training step, for a batch of labeled pairs, mirrored pairs and unlabeled triples. We compute it once on a tiny batch to check that gradients reach the encoder; a real run would loop over SNLI, MultiNLI and the unlabeled data, as in the paper's code ([utahnlp/consistency](https://github.com/utahnlp/consistency)).

In [17]:
def consistency_loss(model, labeled, mirrored_pairs, unlabeled_triples, lambda_sym=1.0, lambda_tran=1.0):
    "L = L_ann + lambda_sym * L_sym + lambda_tran * L_tran, for one batch."
    ## annotation: cross-entropy on labeled pairs (premise, hypothesis, label index in E, C, N)
    logits = nli_logits([p for p, _, _ in labeled], [h for _, h, _ in labeled], model)
    loss = torch.nn.functional.cross_entropy(logits, torch.tensor([y for _, _, y in labeled]), reduction="sum")

    ## symmetry: each pair and its mirror
    if mirrored_pairs:
        fwd = torch.softmax(nli_logits([p for p, _ in mirrored_pairs], [h for _, h in mirrored_pairs], model), -1)
        bwd = torch.softmax(nli_logits([h for _, h in mirrored_pairs], [p for p, _ in mirrored_pairs], model), -1)
        loss = loss + lambda_sym * fuzzy_loss(SYMMETRY, {"C_PH": fwd[:, 1], "C_HP": bwd[:, 1]})

    ## transitivity: the three pairs of each triple
    if unlabeled_triples:
        firsts = [(p, h) for p, h, _ in unlabeled_triples] + [(h, z) for _, h, z in unlabeled_triples] + [(p, z) for p, _, z in unlabeled_triples]
        probs = torch.softmax(nli_logits([a for a, _ in firsts], [b for _, b in firsts], model), -1)
        probs = probs.view(3, len(unlabeled_triples), 3).transpose(0, 1)       ## [triple, pair, label]
        values = {var(l, a, b): probs[:, k, i] for k, (a, b) in enumerate(PAIRS) for i, l in enumerate(LABELS)}
        loss = loss + lambda_tran * fuzzy_loss(TRANSITIVITY, values)
    return loss

label_index = {"entailment": 0, "contradiction": 1, "neutral": 2}
labeled = [(ex["premise"], ex["hypothesis"], label_index[LABEL_NAMES[ex["label"]]]) for ex in snli.select(range(4))]
model.train()
loss = consistency_loss(model, labeled, mirrored_pairs=[(p, h) for p, h, _ in labeled], unlabeled_triples=triples[:4])
loss.backward()
model.eval()
grad_norm = sum(p.grad.norm() ** 2 for p in model.parameters() if p.grad is not None) ** 0.5
print(f"loss {loss.item():.3f}   gradient norm over the encoder {grad_norm:.3f}")
model.zero_grad()

loss 4.236   gradient norm over the encoder 303.503


#### What the paper found

- **Models are inconsistent, whatever their accuracy.** BERT and the LSTM baseline are about equally inconsistent, and more labeled data reduces symmetry violations but not transitivity violations. Given a contradiction on one of a pair and its mirror, BERT disagrees on the other about 60% of the time ($\tau_S$).
- **The losses almost remove inconsistency.** Adding mirrored data (M), unlabeled pairs (U) and triples (T) sharply lowers both $\rho$ and $\tau$ for symmetry and transitivity (Figures 1 and 2). With only 1% of the labels plus the consistency losses, models are more consistent than baselines trained on 100%: label supervision alone does not teach consistency.
- **Accuracy is unchanged.** Test accuracy on SNLI and MultiNLI stays the same or improves slightly (Table 3), which suggests the test sets do not measure consistency at all.
- **Predictions shift to Neutral** on the unlabeled data, which is reasonable for randomly paired captions. Without any labels, the consistency losses alone would push *every* prediction to Neutral, which satisfies all the rules trivially, so some supervision is needed.
- **Constraints help each other.** Training with the symmetry loss alone also reduces transitivity violations, and combining both losses is better than either.

# Notes

- **What we reproduced.** The consistency rules and violation metrics, measured on a public NLI cross-encoder; a translator from `symprog` formulas to product, Gödel and Łukasiewicz fuzzy logic, which recovers the paper's losses exactly; a comparison with `symprog`'s semantic inference; and the combined loss on a tiny batch. We did not fine-tune a model; the paper fine-tunes BERT$_\text{base}$ on SNLI and MultiNLI with 100k unlabeled triples.
- **Our triples are not the paper's.** We build triples from SNLI hypotheses written for the same premise, which are more closely related (and partly labeled) than the paper's random MS COCO captions, so violation rates are not comparable to Table 2.
- **Syntax matters for t-norms.** The translator follows the formula as written, so rewriting a rule (e.g., expanding `Implies` into `Or(Not(...), ...)`) changes the loss and its gradients. This is unlike `symprog`'s solvers, which only see the Boolean function.
- **Numerical stability.** The product residuum divides by the antecedent's value, and the loss takes logarithms, so values are clamped away from 0.

## Next steps

- **Other t-norms.** Repeat the repair experiment with Gödel and Łukasiewicz logic and compare their gradients (see van Krieken et al., 2022, *Analyzing Differentiable Fuzzy Logic Operators*).
- **Semantic loss instead of t-norms.** Replace `fuzzy_loss` with the semantic loss of the same rules (`semantic_loss.ipynb`), conditioning on exactly one label per pair, and compare.
- **Fuzzy logic in symprog.** The translator could become a `symprog` feature, for example a `fuzzy` method on solvers that evaluates the added formulas syntactically in a chosen t-norm.